# 02 — Training (restart-safe)

**But** : entraîner le modèle ResNet50 sur PatchCamelyon.

Ce notebook est **restart-safe** :
- Il cherche automatiquement le dernier checkpoint dans le dossier de sortie.
- S'il en trouve un, il reprend l'entraînement là où il s'était arrêté (optimizer, epoch, historique).
- Sinon, il part de zéro.

> 💡 Si Google Colab coupe le runtime, relancez simplement ce notebook du début : il reprendra tout seul.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/LAAFI_AI
except ImportError:
    pass

%%capture
%pip install -q -r requirements-colab.txt

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import datasets
datasets.logging.set_verbosity_error()

from laafi_ai.cli_train import set_seed
from laafi_ai.config import ExperimentConfig
from laafi_ai.data import PCamDataModule
from laafi_ai.logging_utils import setup_logging
from laafi_ai.model import build_resnet50_classifier, count_trainable_parameters, get_device
from laafi_ai.paths import resolve_project_paths
from laafi_ai.trainer import Trainer

setup_logging()

## 2. Configuration

Modifiez les lignes ci-dessous pour passer du mode *smoke test* à un entraînement complet.

In [ ]:
config = ExperimentConfig.from_yaml('configs/default.yaml')

# =====================================================================
# SMOKE TEST vs ENTRAÎNEMENT COMPLET
# =====================================================================
# Pour un smoke test rapide, décommentez les deux lignes ci-dessous :
# config.data.max_train_samples = 512
# config.data.max_val_samples = 256
#
# Pour un entraînement complet, laissez-les commentées (None = tout).
# =====================================================================

config.data.batch_size = 32
config.training.epochs = 10
config.model.freeze_backbone = True
config.model.unfreeze_layer4 = True
config.optimizer.learning_rate = 1e-4
config.output_dir = 'outputs_finetune_layer4'

set_seed(config.seed)
device = get_device(config.device)
paths = resolve_project_paths(config, project_root=PROJECT_ROOT)

print('Device:', device)
print('Epochs:', config.training.epochs)
print('Output:', paths.output_dir)

## 3. Charger les données

In [ ]:
data_module = PCamDataModule(config.data)
train_loader, val_loader, test_loader = data_module.dataloaders()

print(f'Train: {len(train_loader.dataset):,} images')
print(f'Val:   {len(val_loader.dataset):,} images')

## 4. Construire le modèle et le Trainer

In [ ]:
model = build_resnet50_classifier(config.model)
print(f'Paramètres entraînables: {count_trainable_parameters(model):,}')

trainer = Trainer(model=model, config=config, device=device)

## 5. Reprendre depuis un checkpoint (si disponible)

Le Trainer cherche automatiquement le dernier fichier `.pt` dans le dossier `checkpoints/`.
- S'il en trouve un → il restaure les poids, l'optimizer, l'epoch, et l'historique.
- Sinon → il part de zéro.

In [ ]:
resume_ckpt = trainer.resume_from_checkpoint(paths.checkpoint_dir)

if resume_ckpt is not None:
    print(f'\n✅ Checkpoint trouvé : epoch {resume_ckpt.epoch}, '
          f'best AUC = {resume_ckpt.best_metric:.4f}')
    print(f'   → Reprend à l\'epoch {resume_ckpt.next_epoch}')
else:
    print('\nAucun checkpoint trouvé — entraînement from scratch.')

## 6. Lancer l'entraînement

In [ ]:
history = trainer.fit(train_loader, val_loader, resume_checkpoint=resume_ckpt)

## 7. Résumé de l'entraînement

In [ ]:
import matplotlib.pyplot as plt

if history:
    epochs = [r['epoch'] for r in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, [r['train_loss'] for r in history], label='Train loss')
    ax1.plot(epochs, [r['val_loss'] for r in history], label='Val loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.set_title('Loss')
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, [r['val_auc'] for r in history], label='Val AUC', marker='o')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('AUC')
    ax2.legend()
    ax2.set_title('Validation AUC')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(paths.figures_dir / 'training_curves.png', dpi=150)
    plt.show()
    print('Courbes sauvegardées dans', paths.figures_dir / 'training_curves.png')
else:
    print('Aucune epoch exécutée (probablement déjà terminé).')

---
**Prochain notebook** : `03_eval.ipynb` pour évaluer le meilleur checkpoint sur le test set.